# Sidewalk hazard detector — training

Fine-tunes YOLO26n into the four-class detector the app expects, on a free Colab T4, and exports it as Core ML for iOS.

The model matches the one named in the investigation report. The export format does not: the report says TensorFlow Lite, and this exports Core ML, because the app is iOS-first and TFLite on iOS runs on the CPU or GPU rather than the Neural Engine. That is a deliberate change worth a sentence in the report rather than a silent one. The same trained weights export to TFLite as well if the Android path is needed later — one extra line, no retraining.

**Classes, and where the data comes from.** The order below is not arbitrary: it matches `HAZARD_CLASSES` in `src/types/hazard.ts` exactly, so the model's class indices are the app's class indices and nothing has to be remapped in Swift.

| # | Class | Source | Coverage |
|---|---|---|---|
| 0 | `pothole` | RDD2022, damage type D40 | Excellent — thousands of instances |
| 1 | `slippery-surface` | Roboflow puddle / wet-surface sets | **Weak** — a couple of thousand images at best |
| 2 | `broken-tactile-paving` | RDD2022, types D00/D10/D20 (cracks) | Good, but see the note below |
| 3 | `pathway-obstruction` | COCO subset, all collapsed to one class | Excellent — COCO is enormous |

**Two honest caveats, worth writing into the report rather than hiding.**

Class 2 is trained on road cracking, not on tactile paving specifically. That is defensible because the app's own setting is labelled *"Broken Tactile Paving / Uneven Surface"* — cracked and broken pavement is the hazard, and it is what the walker needs warning about. But a model trained this way will not reliably recognise *intact* tactile paving as distinct from a cracked slab. If tactile paving specifically matters, add a dedicated dataset (see the search terms in the tactile cell) or collect your own.

Class 1 is the weak one. There is no substantial public dataset of slippery surfaces with bounding boxes, because "slippery" is not visible — standing water is, and that is what these datasets actually label. Expect this class to underperform the other three, report it as a limitation, and consider collecting a few hundred of your own images if it matters to your marks.

**Free tier survival.** Sessions drop without warning. Everything is written to Drive, and the training cell is resume-safe — if you get disconnected, run the resume cell and it picks up from the last epoch rather than starting over.

## 1. Runtime

Set **Runtime → Change runtime type → T4 GPU** before running anything. If the check below says CUDA is unavailable, you are on a CPU runtime and training will take days rather than hours.

In [ ]:
!nvidia-smi

import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Sun Aug 16 18:20:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Weights and results live on Drive so a dropped session costs minutes, not hours.
PROJECT = Path('/content/drive/MyDrive/hazard-detector')
PROJECT.mkdir(parents=True, exist_ok=True)

# The dataset itself stays on local disk - it is large, and Drive I/O during
# training is slow enough to bottleneck the GPU.
DATA = Path('/content/data')
DATA.mkdir(parents=True, exist_ok=True)

print('project:', PROJECT)
print('data   :', DATA)

Mounted at /content/drive
project: /content/drive/MyDrive/hazard-detector
data   : /content/data


In [ ]:
# Deliberately short. Ultralytics brings almost everything; coremltools does the
# export; roboflow fetches the puddle sets.
#
# Note what is *not* here: fiftyone, the usual way to pull a COCO subset. It
# installs MongoDB and its own Pillow pin, and that pin collides with the one
# Colab ships - leaving PIL half-upgraded and every image library broken with
# "cannot import name '_Ink' from 'PIL._typing'". COCO is served over plain
# HTTP, so section 4 fetches it directly and the whole dependency goes away.
!pip -q install -U ultralytics coremltools roboflow

import ultralytics
ultralytics.checks()

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.3/112.6 GB disk)


## 2. Class definitions

Change these and you change the model's contract with the app. The indices must stay aligned with `HAZARD_CLASSES`.

In [ ]:
CLASSES = ['pothole', 'slippery-surface', 'broken-tactile-paving', 'pathway-obstruction']
POTHOLE, SLIPPERY, UNEVEN, OBSTRUCTION = range(4)

# Where each converted dataset gets written, one directory per source.
STAGE = DATA / 'stage'
STAGE.mkdir(parents=True, exist_ok=True)

import collections, random, shutil
random.seed(0)

def write_label(dst_dir, stem, rows):
    """rows: list of (cls, xc, yc, w, h) already normalised to 0-1."""
    with open(dst_dir / (stem + '.txt'), 'w') as fh:
        for cls, xc, yc, w, h in rows:
            fh.write(f'{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n')

def stage_dirs(name):
    images = STAGE / name / 'images'
    labels = STAGE / name / 'labels'
    images.mkdir(parents=True, exist_ok=True)
    labels.mkdir(parents=True, exist_ok=True)
    return images, labels

## 3. RDD2022 — potholes and cracked surface

47,420 road images from six countries, annotated in Pascal VOC XML with damage types D00 (longitudinal crack), D10 (transverse crack), D20 (alligator crack) and D40 (pothole). Two of our four classes come from here.

The cell below lists the files from the figshare API rather than hard-coding a download URL, because figshare's per-file URLs change between versions.

**It comes as one 13 GB archive for all six countries**, not one file per country, so there is no way to download less. And each country inside it is *itself* a zip, so extraction happens twice — the first pass yields only more archives, which looks like failure and is not.

Four countries are extracted: Japan, Czech, India and the United States. The choice is driven by class balance rather than by size — potholes are the scarcest of the four hazard classes, and India and the US carry the most of them.

Budget roughly ten minutes for the download. It is the slowest cell in the notebook.

> Arya, D., Maeda, H., Kumar Ghosh, S., Toshniwal, D., & Sekimoto, Y. (2022). *RDD2022: A multi-national image dataset for automatic Road Damage Detection.* [doi:10.6084/m9.figshare.21431547](https://doi.org/10.6084/m9.figshare.21431547)

In [ ]:
import requests

meta = requests.get('https://api.figshare.com/v2/articles/21431547').json()
print(meta['title'], '\n')
for f in meta['files']:
    print(f"{f['name']:34s} {f['size'] / 1e9:6.2f} GB")

FILES = {f['name']: f['download_url'] for f in meta['files']}

RDD2022 - The multi-national Road Damage Dataset released through CRDDC'2022 

RDD2022_released_through_CRDDC2022.zip  13.26 GB
Directory_Structure_CRDDC_RDD2022.txt   0.00 GB
File_List_CRDDC_RDD2022.txt          0.00 GB
label_map.pbtxt                      0.00 GB


In [ ]:
import subprocess

# RDD2022 is distributed as ONE archive containing all six countries, not as one
# archive per country. So the country choice happens at extraction, not at
# download: there is no way to fetch less than the whole 13 GB.
#
# These four, and not the other two. Potholes are the scarcest of the four
# hazard classes and the one this app is most obviously about, so the countries
# are chosen to feed that class: India and the United States are pothole-rich,
# Japan and Czech add road variety. Norway is left out deliberately - at 10.6 GB
# it is larger than the other five together and would not fit alongside the rest
# on a free runtime.
COUNTRIES = ['Japan', 'Czech', 'India', 'United_States']

zips = [f for f in meta['files'] if f['name'].lower().endswith('.zip')]
if not zips:
    raise SystemExit(
        'No .zip in the figshare listing above. The release layout has changed - '
        'set the archive by hand from the printed file names.'
    )

archive = max(zips, key=lambda f: f['size'])
print(f"archive: {archive['name']}  ({archive['size'] / 1e9:.2f} GB)")

RDD = DATA / 'rdd'
RDD.mkdir(parents=True, exist_ok=True)
archive_path = str(RDD / archive['name'])
archive_url = archive['download_url']

# Checked by size, not by existence. A download interrupted at 9 of 13 GB leaves
# a file that exists, so an existence check would skip straight past it to an
# unzip that fails on a truncated archive - or worse, half-succeeds.
expected = archive['size']
have = Path(archive_path).stat().st_size if Path(archive_path).exists() else 0

if have == expected:
    print('already downloaded, size matches')
else:
    if have:
        print(f'incomplete: {have / 1e9:.2f} of {expected / 1e9:.2f} GB - starting over')
        Path(archive_path).unlink()
    print('downloading - the long step, expect several minutes')
    !wget -q --show-progress -O "{archive_path}" "{archive_url}"

    got = Path(archive_path).stat().st_size if Path(archive_path).exists() else 0
    if got != expected:
        raise SystemExit(
            f'Download finished at {got / 1e9:.2f} GB but figshare reports '
            f'{expected / 1e9:.2f} GB. Re-run this cell.'
        )

archive: RDD2022_released_through_CRDDC2022.zip  (13.26 GB)
downloading - the long step, expect several minutes
/content/data/rdd/R 100%[===================>]  12.35G  20.0MB/s    in 10m 39s 


In [ ]:
# What the archive actually contains, before committing to extracting any of
# it. The country names have to appear somewhere in these paths for the filter
# below to work, and looking is cheaper than guessing - which is precisely the
# mistake that cost the first training run two of its four classes.
listing = subprocess.run(['unzip', '-l', archive_path], capture_output=True, text=True).stdout
print('\n'.join(listing.splitlines()[:15]))

Archive:  /content/data/rdd/RDD2022_released_through_CRDDC2022.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
160193541  2022-08-10 16:36   RDD2022/China_Drone.zip
192030116  2022-08-10 16:36   RDD2022/China_MotorBike.zip
257117218  2022-08-10 16:39   RDD2022/Czech.zip
526709145  2022-08-10 16:42   RDD2022/India.zip
1072598857  2022-08-10 16:44   RDD2022/Japan.zip
10611186814  2022-08-10 16:59   RDD2022/Norway.zip
444335958  2022-08-10 16:39   RDD2022/United_States.zip
---------                     -------
13264171649                     7 files


In [ ]:
# Zips inside a zip. The outer archive holds one archive per country, so the
# images and XML sit two levels down and the country filter has to be applied
# twice - once to pull out the right country archives, once to open them.
#
# The intermediate state is worth naming, because it looks like failure: after
# the first extraction there is not a single .xml on disk, only more .zip files.
patterns = [f'*{country}*' for country in COUNTRIES]
print('extracting country archives:', patterns)
subprocess.run(['unzip', '-q', '-o', archive_path, *patterns, '-d', str(RDD)], check=False)

def inner_archives():
    return [p for p in RDD.rglob('*.zip') if p.resolve() != Path(archive_path).resolve()]

if not inner_archives():
    print('country filter matched nothing - extracting the whole outer archive')
    subprocess.run(['unzip', '-q', '-o', archive_path, '-d', str(RDD)], check=False)

wanted = [
    p for p in inner_archives()
    if any(country.lower() in p.name.lower() for country in COUNTRIES)
]
if not wanted:
    raise SystemExit(
        'No country archive matched ' + str(COUNTRIES) + '. Found: ' +
        str([p.name for p in inner_archives()])
    )

print('opening:', [p.name for p in wanted])
for country_zip in wanted:
    subprocess.run(['unzip', '-q', '-o', str(country_zip), '-d', str(RDD)], check=False)

if not any(RDD.rglob('*.xml')):
    raise SystemExit('Still no annotations after opening the country archives.')

# Reclaim the disk taken by country archives that are not being used - Norway
# alone is 10.6 GB, and a free runtime does not have room to spare for it.
for spare in inner_archives():
    if spare not in wanted:
        spare.unlink()

!du -sh "{RDD}"

extracting country archives: ['*Japan*', '*Czech*', '*India*', '*United_States*']
opening: ['Japan.zip', 'India.zip', 'United_States.zip', 'Czech.zip']
17G	/content/data/rdd


In [ ]:
import xml.etree.ElementTree as ET

# D40 is a pothole. The three crack types all describe a broken, uneven walking
# surface, which is the hazard the app's "Broken Tactile Paving / Uneven Surface"
# setting covers - so they collapse into one class rather than three the user
# would never distinguish between.
RDD_MAP = {'D40': POTHOLE, 'D00': UNEVEN, 'D10': UNEVEN, 'D20': UNEVEN}

# Cracks outnumber potholes heavily in RDD2022. Left alone the model learns to
# call everything a crack, so crack-only images are subsampled while every image
# containing a pothole is kept.
CRACK_ONLY_KEEP = 0.35

images_out, labels_out = stage_dirs('rdd')
counts = collections.Counter()
kept = skipped = 0

# Every image in the extracted archive, indexed by filename stem.
#
# Built by walking the whole tree rather than by assuming the layout, because
# the annotation and image directories sit at different depths in different
# releases of this dataset - and a path assumption that misses simply stages
# nothing, silently, which is exactly the failure this replaces.
image_index = {}
for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG'):
    for path in RDD.rglob(pattern):
        image_index.setdefault(path.stem, path)

annotations = list(RDD.rglob('*.xml'))
print('found', len(annotations), 'annotations and', len(image_index), 'images')
if not annotations or not image_index:
    raise SystemExit(
        f'RDD2022 extracted but no annotations or images were found under {RDD}. '
        f'Run  !find "{RDD}" -maxdepth 4 -type d  to see what the archive actually contains.'
    )

for xml_path in annotations:
    root = ET.parse(xml_path).getroot()
    size = root.find('size')
    W, H = int(size.find('width').text), int(size.find('height').text)
    if W == 0 or H == 0:
        continue

    rows = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in RDD_MAP:
            continue
        box = obj.find('bndbox')
        x1, y1 = float(box.find('xmin').text), float(box.find('ymin').text)
        x2, y2 = float(box.find('xmax').text), float(box.find('ymax').text)
        x1, x2 = sorted((max(0, x1), min(W, x2)))
        y1, y2 = sorted((max(0, y1), min(H, y2)))
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        rows.append((RDD_MAP[name], ((x1 + x2) / 2) / W, ((y1 + y2) / 2) / H,
                     (x2 - x1) / W, (y2 - y1) / H))

    if not rows:
        continue
    if POTHOLE not in {r[0] for r in rows} and random.random() > CRACK_ONLY_KEEP:
        skipped += 1
        continue

    stem = xml_path.stem
    src_img = image_index.get(stem)
    if src_img is None:
        continue

    shutil.copy(src_img, images_out / src_img.name)
    write_label(labels_out, stem, rows)
    counts.update(CLASSES[r[0]] for r in rows)
    kept += 1

print('images kept:', kept, ' crack-only images dropped:', skipped)
print('instances  :', dict(counts))

if kept == 0:
    raise SystemExit(
        'RDD staged zero images - pothole and broken-tactile-paving would both be '
        'empty. Do not carry on to training.'
    )

found 25846 annotations and 32341 images
images kept: 8010  crack-only images dropped: 8990
instances  : {'broken-tactile-paving': 12618, 'pothole': 5762}


## 4. COCO subset — pathway obstruction

Nothing needs training from scratch here. COCO already contains, labelled and in quantity, exactly the things that block a pavement: people, bicycles, parked cars, bins, benches, dogs. They all collapse into one class, because the app does not care *what* is in the way — the guidance is the same either way.

`COCO_MAX_IMAGES` is the knob that keeps this class from swamping the other three. Four thousand is roughly the size of the RDD contribution.

Only the annotations are downloaded in bulk; the images are fetched individually, and only the ones actually chosen — about 650 MB rather than the full 19 GB of COCO train2017.

In [ ]:
import json, gc, urllib.request
from concurrent.futures import ThreadPoolExecutor

OBSTRUCTION_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck',
    'bench', 'chair', 'potted plant', 'dog', 'suitcase', 'backpack',
    'fire hydrant', 'stop sign', 'parking meter',
]
COCO_MAX_IMAGES = 4000

COCO = DATA / 'coco'
COCO.mkdir(parents=True, exist_ok=True)
ann_zip = COCO / 'annotations_trainval2017.zip'
ann_json = COCO / 'annotations' / 'instances_train2017.json'

if not ann_json.exists():
    if not ann_zip.exists():
        !wget -q --show-progress -O "{ann_zip}" http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o "{ann_zip}" 'annotations/instances_train2017.json' -d "{COCO}"

print('parsing annotations - this takes a minute and a few GB of RAM')
with open(ann_json) as fh:
    raw = json.load(fh)

/content/data/coco/ 100%[===================>] 241.19M  59.7MB/s    in 4.5s    
parsing annotations - this takes a minute and a few GB of RAM


In [ ]:
name_to_id = {c['name']: c['id'] for c in raw['categories']}
wanted = {name_to_id[n] for n in OBSTRUCTION_CLASSES if n in name_to_id}
missing = [n for n in OBSTRUCTION_CLASSES if n not in name_to_id]
if missing:
    print('not COCO categories, ignored:', missing)

# image id -> its boxes, keeping only the categories we asked for. Everything
# else in the image is dropped: these images are full of labelled sofas and
# sandwiches, and teaching those as pavement obstructions would be worse than
# not training the class at all.
boxes = collections.defaultdict(list)
for ann in raw['annotations']:
    if ann['category_id'] in wanted and not ann.get('iscrowd', 0):
        boxes[ann['image_id']].append(ann['bbox'])

# Not called "meta" - that name is already the figshare response from section 3,
# and a notebook shares one namespace across every cell.
coco_images = {im['id']: im for im in raw['images'] if im['id'] in boxes}

# The parsed JSON is several GB and is not needed again.
del raw
gc.collect()

chosen = sorted(coco_images)
random.Random(0).shuffle(chosen)
chosen = chosen[:COCO_MAX_IMAGES]
print(len(boxes), 'images contain a wanted class;', len(chosen), 'selected')

80768 images contain a wanted class; 4000 selected


In [ ]:
images_out, labels_out = stage_dirs('coco')

def fetch(image_id):
    info = coco_images[image_id]
    W, H = info['width'], info['height']
    destination = images_out / info['file_name']

    rows = []
    for x, y, w, h in boxes[image_id]:
        if w / W < 0.02 or h / H < 0.02:   # specks the detector cannot use
            continue
        rows.append((OBSTRUCTION, (x + w / 2) / W, (y + h / 2) / H, w / W, h / H))
    if not rows:
        return 0

    if not destination.exists():
        try:
            urllib.request.urlretrieve(info['coco_url'], destination)
        except Exception:
            return 0

    write_label(labels_out, destination.stem, rows)
    return 1

# Threaded because this is entirely network-bound - sequentially it would take
# well over an hour.
with ThreadPoolExecutor(max_workers=16) as pool:
    kept = sum(pool.map(fetch, chosen))

print('coco images staged:', kept)

coco images staged: 3956


## 5. Puddles and wet surface — slippery

The weak class. These are small community datasets rather than benchmarks, so quality varies and you should look at a few images before trusting them.

Get a free key from [roboflow.com](https://roboflow.com) → Settings → API keys. Candidates worth pulling, largest first:

- `case-ows0c/water-and-wet-surface` — ~1,800 images
- `icews/water-puddles-f5tdo` — ~400 images
- `puddle-water-detection/pkl-hkosw` — ~250 images

Search [universe.roboflow.com](https://universe.roboflow.com/search?q=class%3Apuddle) for more. If you later collect your own wet-pavement photos, they go in the same staging directory and need no other change.

In [ ]:
ROBOFLOW_KEY = 'jGE8gX17AfNuogH1nDvf'   # paste your key

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_KEY)

# (workspace, project) pairs, taken from the universe URL:
#   universe.roboflow.com/<workspace>/<project>
SOURCES = [
    ('case-ows0c', 'water-and-wet-surface'),
    ('icews', 'water-puddles-f5tdo'),
]

downloaded = []
for workspace, project_id in SOURCES:
    try:
        project = rf.workspace(workspace).project(project_id)
        # Version ids come back as strings, and sometimes as a full
        # "workspace/project/3" path. Comparing those with max() sorts them
        # lexicographically, which quietly picks v9 over v10 - so the trailing
        # number is pulled out and compared as an integer.
        version = max(int(str(v.version).rstrip('/').split('/')[-1]) for v in project.versions())
        out = DATA / 'wet' / project_id
        project.version(version).download('yolov8', location=str(out))
        downloaded.append(out)
        print('ok:', project_id, 'v' + str(version))
    except Exception as exc:
        print('skipped', project_id, '-', exc)

print(downloaded)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/wet/water-and-wet-surface in yolov8:: 100%|██████████| 3623/3623 [00:00<00:00, 7680.11it/s]

ok: water-and-wet-surface v2
loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to /content/data/wet/water-puddles-f5tdo in yolov8:: 100%|██████████| 812/812 [00:00<00:00, 8151.63it/s]

ok: water-puddles-f5tdo v1
[PosixPath('/content/data/wet/water-and-wet-surface'), PosixPath('/content/data/wet/water-puddles-f5tdo')]


In [ ]:
# See the comment where this is applied, below.
MAX_WET_BOXES_PER_IMAGE = 6

# Tolerates the Roboflow cell having been skipped entirely - the other three
# classes still train, this one is simply absent.
downloaded = globals().get('downloaded', [])

images_out, labels_out = stage_dirs('wet')
kept = 0

# Every class in these datasets - "puddle", "water", "wet surface" - means the
# same thing to the app, so all of them collapse to one index.
for root in downloaded:
    for label_file in root.rglob('labels/*.txt'):
        image_file = None
        for ext in ('.jpg', '.jpeg', '.png'):
            candidate = label_file.parent.parent / 'images' / (label_file.stem + ext)
            if candidate.exists():
                image_file = candidate
                break
        if image_file is None:
            continue

        rows = []
        for line in label_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            xc, yc, w, h = (float(p) for p in parts[1:5])
            # Same minimum size the COCO staging applies.
            if w < 0.02 or h < 0.02:
                continue
            rows.append((SLIPPERY, xc, yc, w, h))
        if not rows:
            continue

        # The largest few only.
        #
        # These sets annotate wet ground as a scattering of adjacent patches -
        # thirteen boxes per image on average, against four or five for every
        # other source. Left alone that one class contributes more instances
        # than the other three together, and the model learns to see water
        # everywhere. Keeping the biggest patches keeps what a walker would
        # actually notice and drops the fragmentation.
        rows.sort(key=lambda r: r[3] * r[4], reverse=True)
        rows = rows[:MAX_WET_BOXES_PER_IMAGE]

        # Names collide across datasets, so they are prefixed.
        stem = root.name + '_' + label_file.stem
        shutil.copy(image_file, images_out / (stem + image_file.suffix))
        write_label(labels_out, stem, rows)
        kept += 1

print('wet-surface images staged:', kept)

wet-surface images staged: 2206


## 6. Merge, split, and look at the balance

The split is by image, 80/20, with a fixed seed so a re-run gives the same split — otherwise validation scores drift between runs for reasons that have nothing to do with the model.

Read the instance counts carefully. If one class has an order of magnitude more instances than another, the model will quietly learn to favour it, and mAP averaged over classes will hide that.

In [ ]:
import yaml

DATASET = DATA / 'hazards'

# Cleared before rebuilding, which matters the moment you add a source and run
# this again. Left in place, the previous split's files stay exactly where they
# were while the new split reassigns them - so an image can sit in train and val
# at once. Nothing errors; the validation scores simply come out flattering,
# because the model is being marked on images it was trained on.
if DATASET.exists():
    shutil.rmtree(DATASET)

for split in ('train', 'val'):
    (DATASET / split / 'images').mkdir(parents=True, exist_ok=True)
    (DATASET / split / 'labels').mkdir(parents=True, exist_ok=True)

pairs = []
for stage in STAGE.iterdir():
    if not (stage / 'images').is_dir():
        continue
    for image in (stage / 'images').iterdir():
        label = stage / 'labels' / (image.stem + '.txt')
        if label.exists():
            pairs.append((image, label))

random.Random(0).shuffle(pairs)
cut = int(len(pairs) * 0.8)

for split, chunk in (('train', pairs[:cut]), ('val', pairs[cut:])):
    for image, label in chunk:
        shutil.copy(image, DATASET / split / 'images' / image.name)
        shutil.copy(label, DATASET / split / 'labels' / label.name)

data_yaml = DATASET / 'hazards.yaml'
data_yaml.write_text(yaml.safe_dump({
    'path': str(DATASET),
    'train': 'train/images',
    'val': 'val/images',
    'names': {i: name for i, name in enumerate(CLASSES)},
}, sort_keys=False))

print(data_yaml.read_text())
print('train:', cut, ' val:', len(pairs) - cut)

path: /content/data/hazards
train: train/images
val: val/images
names:
  0: pothole
  1: slippery-surface
  2: broken-tactile-paving
  3: pathway-obstruction

train: 11337  val: 2835


In [ ]:
totals = collections.Counter()

for split in ('train', 'val'):
    per_class = collections.Counter()
    images_with = collections.Counter()
    for label in (DATASET / split / 'labels').iterdir():
        present = set()
        for line in label.read_text().strip().splitlines():
            if line:
                cls = int(line.split()[0])
                per_class[CLASSES[cls]] += 1
                present.add(CLASSES[cls])
        images_with.update(present)

    print(f'--- {split} ---')
    for name in CLASSES:
        print(f'  {name:24s} {per_class[name]:6d} instances in {images_with[name]:5d} images')
    totals.update(per_class)

# A hard stop rather than a warning. A class with no data does not fail during
# training - it trains perfectly happily and simply never predicts that class,
# and the first sign of it is a validation table two rows short, hours later.
empty = [name for name in CLASSES if totals[name] == 0]
if empty:
    raise SystemExit(
        'No data at all for: ' + ', '.join(empty) +
        '. Go back and fix the staging cell that feeds those classes before training.'
    )
print('\nall four classes have data - safe to train')

--- train ---
  pothole                    4687 instances in  2568 images
  slippery-surface           9169 instances in  1781 images
  broken-tactile-paving     10004 instances in  5302 images
  pathway-obstruction       14204 instances in  3171 images
--- val ---
  pothole                    1075 instances in   622 images
  slippery-surface           2186 instances in   425 images
  broken-tactile-paving      2614 instances in  1374 images
  pathway-obstruction        3727 instances in   785 images

all four classes have data - safe to train


## 7. Train

YOLO26n — the nano variant, and the model the investigation report already names. On a phone that is the only sensible size: it runs in a few milliseconds on the Neural Engine, where a small or medium model would drain the battery and drop frames while barely improving on hazards this large in frame.

YOLO26 is end-to-end by default, meaning its detection head emits final boxes with no non-maximum suppression step at all. That matters more on a phone than the accuracy figures do: NMS is post-processing that runs on the CPU every frame, and removing it takes work off the same core that is holding the AR tracking together.

Roughly 3 minutes per epoch on a T4 at this dataset size, so 50 epochs is around two and a half hours. That is longer than a free session is guaranteed to last, which is what the resume cell below is for.

**If you get disconnected**, what to re-run depends on which of two different things happened.

If the *browser* dropped but the runtime survived — Colab shows "Reconnect" and the epochs keep ticking — do nothing. Reconnect and watch.

If the *runtime* was recycled, the weights on Drive survived but the dataset on local disk did not. Re-run section 1, then **sections 2 to 6 to rebuild the dataset**, and only then the resume cell. Rebuilding is not optional: the seeds are fixed so you get the same split back, and without it resume has no data to read.

Either way, never re-run the training cell above to "restart" — it begins from epoch zero and overwrites the checkpoint. And a run that has *finished* cannot be resumed at all; start a new one under a new `RUN_NAME`.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'hazard-yolo26n'

model = YOLO('yolo26n.pt')          # COCO-pretrained; training from scratch on
                                    # this little data would be far worse
results = model.train(
    data=str(data_yaml),
    # Fewer epochs than you might expect, because the dataset roughly doubled
    # when two more countries were added. Epoch time scales with images, so 80
    # epochs would now run past five hours - well beyond what a free session
    # survives - while 50 over twice the data sees more examples than 80 over
    # half of it did. patience below may stop it earlier still.
    epochs=50,
    imgsz=640,
    batch=16,                       # fits a T4 comfortably at 640
    workers=2,                      # Colab throttles above this
    patience=20,                    # stop if val stops improving
    close_mosaic=10,                # last 10 epochs without mosaic, so the model
                                    # finishes on images shaped like real frames
    project=str(PROJECT / 'runs'),
    name=RUN_NAME,
    exist_ok=True,
    seed=0,
    plots=True,
)

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/hazards/hazards.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=hazard-yolo26n, nbs=64, nms=

In [ ]:
# RESUME - safe to leave in a "Run all". It works out for itself whether
# resuming is the right thing to do and skips quietly when it is not, so it can
# sit between the training cell and validation without doing harm either way.
import yaml
from ultralytics import YOLO

RUN_NAME = 'hazard-yolo26n'
resume_data = DATA / 'hazards' / 'hazards.yaml'
run_dir = PROJECT / 'runs' / RUN_NAME
resume_weights = run_dir / 'weights' / 'last.pt'
results_csv = run_dir / 'results.csv'
args_file = run_dir / 'args.yaml'

completed = (
    max(0, len(results_csv.read_text().strip().splitlines()) - 1)
    if results_csv.exists() else 0
)
planned = yaml.safe_load(args_file.read_text()).get('epochs') if args_file.exists() else None
print(f'checkpoint: {completed} epochs done' + (f' of {planned}' if planned else ''))

# Each of these is a reason NOT to resume, and each is a silent disaster if
# ignored rather than an error:
#
# - no checkpoint: nothing to continue from.
# - no dataset: resume reads the dataset path out of args.yaml, and when that
#   path is gone - which it always is after the VM is recycled, since the data
#   lives on local disk - Ultralytics does not fail. It falls back to its
#   defaults and trains on coco8, an eight-image smoke-test set, for 100 epochs,
#   overwriting your weights as it goes.
# - already finished: Ultralytics starts a brand new run on default settings,
#   destroying the weights that just took hours to produce.
if not resume_weights.exists():
    print('nothing to resume - no checkpoint yet. Skipping.')
elif not resume_data.exists():
    print('nothing to resume - the dataset is not on this runtime, and resuming')
    print('without it would silently train on coco8. Re-run sections 2 to 6 first.')
elif planned and completed >= planned:
    print(f'nothing to resume - all {planned} epochs are already done. Skipping to validation.')
else:
    print('resuming...')
    model = YOLO(str(resume_weights))
    results = model.train(resume=True)

checkpoint: 31 epochs done of 50
resuming...
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/hazards/hazards.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/hazard-detector/runs/h

## 8. Validate

Look at per-class numbers, not the headline mAP. A strong average across four classes can hide one class being close to useless — which, given where the data came from, is the expected failure mode for `slippery-surface`.

The confusion matrix written into the run directory is worth putting in the report.

In [ ]:
best = YOLO(str(PROJECT / 'runs' / RUN_NAME / 'weights' / 'best.pt'))
metrics = best.val(data=str(data_yaml), imgsz=640, plots=True)

# class_result(i) is indexed by position among the classes that actually appear
# in the validation set, NOT by class id. Those are the same list only when
# every class has data - so indexing it with a class id silently prints one
# class's numbers under another class's name, and raises IndexError once enough
# classes are missing. ap_class_index is the mapping back to real ids.
present = list(metrics.ap_class_index)

print(f"{'class':24s} {'P':>7s} {'R':>7s} {'mAP50':>7s} {'mAP50-95':>9s}")
for position, class_id in enumerate(present):
    p, r, map50, map5095 = metrics.class_result(position)
    print(f'{CLASSES[class_id]:24s} {p:7.3f} {r:7.3f} {map50:7.3f} {map5095:9.3f}')

for class_id, name in enumerate(CLASSES):
    if class_id not in present:
        print(f'{name:24s} {"NO DATA - this class was never trained":>7s}')

print(f"\n{'overall':24s} {'':7s} {'':7s} {metrics.box.map50:7.3f} {metrics.box.map:9.3f}")

Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1871.2±908.8 MB/s, size: 95.8 KB)
val: Scanning /content/data/hazards/val/labels.cache... 2835 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2835/2835 1.1Git/s 0.0s
val: /content/data/hazards/val/images/Japan_001971.jpg: 1 duplicate labels removed
val: /content/data/hazards/val/images/Japan_012446.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 178/178 6.0it/s 29.8s
                   all       2835       9600      0.511      0.455      0.468      0.232
               pothole        622       1075      0.462      0.483      0.464      0.195
      slippery-surface        425       2186      0.437      0.186      0.243     0.0883
 broken-tactile-paving       1374       2612     

## 9. Export to Core ML

The two export arguments below need explaining together, because they look like they contradict the point of YOLO26.

YOLO26's end-to-end head emits a raw `(1, 300, 6)` tensor — 300 candidate rows of `[x1, y1, x2, y2, confidence, class]`, already deduplicated. Apple's Vision framework does not recognise that as a detector. It returns decoded `VNRecognizedObjectObservation`s, with labels and normalised boxes, only for models carrying Core ML's own NMS metadata — so exporting end-to-end means writing the decode by hand in Swift.

`end2end=False` with `nms=True` therefore trades YOLO26's NMS-free inference for Core ML's built-in NMS layer, and buys a Swift integration of about fifteen lines that cannot get the box maths wrong. Core ML's NMS runs natively rather than in application code, so the cost is small.

**If you would rather keep the end-to-end head**, export with `nms=False, end2end=True` and decode the tensor yourself: filter rows by confidence, scale the four coordinates by the input size, and that is the whole algorithm — no NMS to write, because there is none to do. It is the faster path and the more modern one; it is just more Swift.

Either way the boxes end up as `NormalizedBoundingBox` in the app, with one adjustment for the Vision route: Vision's origin is bottom-left and the app's is top-left, so `y` needs flipping.

In [ ]:
best = YOLO(str(PROJECT / 'runs' / RUN_NAME / 'weights' / 'best.pt'))

EXPORT_ARGS = dict(
    format='coreml',
    imgsz=640,
    end2end=False,  # swap YOLO26's NMS-free head for the classic one, so that...
    nms=True,       # ...Core ML's NMS layer makes this a Vision-native detector
    half=True,      # fp16 - half the size, and the Neural Engine runs fp16 anyway
)

try:
    exported = best.export(**EXPORT_ARGS)
except TypeError as exc:
    # Ultralytics rejects export arguments it does not recognise, and end2end is
    # specific to the YOLO26 head. If this version has not got it, the classic
    # head is the default anyway and nms=True alone gives the same result.
    print('end2end not accepted by this Ultralytics version, retrying without it:', exc)
    EXPORT_ARGS.pop('end2end')
    exported = best.export(**EXPORT_ARGS)

print('exported to:', exported)

import shutil
destination = PROJECT / 'HazardDetector.mlpackage'
if destination.exists():
    shutil.rmtree(destination)
shutil.copytree(exported, destination)
print('copied to Drive:', destination)

!du -sh "{destination}"

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 146 layers, 2,495,864 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (5.1 MB)

CoreML: starting export with coremltools 9.0...


Running MIL default pipeline:   8%|▊         | 8/95 [00:00<00:01, 71.17 passes/s]Output, '1324', of the source model, has been renamed to 'var_1324' in the Core ML model.
Output, '1326', of the source model, has been renamed to 'var_1326' in the Core ML model.
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 69.70 passes/s]


CoreML: starting pipeline with coremltools 9.0...
CoreML: pipeline success
CoreML: export success ✅ 24.1s, saved as '/content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.mlpackage' (4.8 MB)

Export complete (24.8s)
Results saved to /content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.mlpackage
Predict:         yolo predict task=detect model=/content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.mlpackage imgsz=640 quantize=16
Validate:        yolo val task=detect model=/content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.mlpackage imgsz=640 data=/content/data/hazards/hazards.yaml quantize=16 
Visualize:       https://netron.app
exported to: /content/drive/MyDrive/hazard-detector/runs/hazard-yolo26n/weights/best.mlpackage
copied to Drive: /content/drive/MyDrive/hazard-detector/HazardDetector.mlpackage
4.8M	/content/drive/MyDrive/hazard-detector/HazardDetector.mlpackage


In [ ]:
# Zip it for download - .mlpackage is a directory, so the browser cannot fetch
# it directly.
!cd "{PROJECT}" && zip -qr HazardDetector.mlpackage.zip HazardDetector.mlpackage

from google.colab import files
files.download(str(PROJECT / 'HazardDetector.mlpackage.zip'))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. Installing it in the app

The app side is already built and waiting. There is exactly one step:

**Put `HazardDetector.mlpackage` in `modules/ar-geospatial/ios/`, commit it, and push.** CI does the rest.

Unzip the download first — it must go in as a directory named `HazardDetector.mlpackage`, not as the `.zip`. It is about 6 MB, which is fine for plain git; no LFS needed.

**Why that directory and not `assets/`.** The `ios/` project is generated by `expo prebuild` and thrown away on every build, so a file added to it by hand does not survive. The module's podspec lists the model as a resource instead, and CocoaPods copies it into the app bundle on every build. The app compiles it to `.mlmodelc` on first launch and caches that, so the one-off compile cost is paid once per install rather than per launch.

**What is already wired**, for reference:

- `HazardDetector.swift` — loads the model, runs it through Vision at 10 Hz, converts boxes from Vision's bottom-left origin to the app's top-left
- `HazardCameraView.swift` — the Hazard Detection screen's camera, AVFoundation only
- `ARGeospatialView.swift` — runs the same detector over ARKit's frames on the AR Navigation screen
- `useHazardDetections` — filters by confidence and by the hazard types left switched on in Settings

Both screens share one detector, so the same pothole is flagged identically whichever tab you are on.

**Class names are the contract.** Vision returns the label string the model was trained with, and the app checks it against `HAZARD_CLASSES`. That is why `CLASSES` at the top of this notebook must not be renamed — a mismatch does not crash, it silently drops every detection.

**Before the model exists**, the app builds and runs fine: the pod resource is simply absent, and the Hazard screen says "Detection unavailable" instead of looking like a camera pointed at flawless pavement.